# 過去CSVのクリーニングとGCSアップロード

旧13の移植。新収集分は最初からスキーマ統一済みのため、**過去データの一括整備と検証専用**。

手順: Phase1 棚卸し（読み取りのみ）→ Phase2 整形（dry_run → 本実行）→ 整合性チェック → GCSアップロード

In [ ]:
#@title 🔧 セットアップ（パッケージinstall + Driveマウント）
GITHUB_OWNER = "asmrt"  # ← GitHubユーザー名に変更

from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q "git+https://{token}@github.com/{GITHUB_OWNER}/unofficial_sixfonia_analytics.git"
drive.mount("/content/drive")

from sixfonia_analytics import config, maintain

In [ ]:
#@title 🔎 Phase 1: 列の棚卸し（読み取りのみ・破壊なし）
for name in config.CHANNEL_NAMES:
    maintain.inventory_columns(config.channel_dir(name))

In [ ]:
#@title 🧪 Phase 2: 整形ドライラン（変更なし・補完予定の確認）
for name in config.CHANNEL_NAMES:
    maintain.align_schema(config.channel_dir(name), dry_run=True)

⚠️ **次のセルは元ファイルを上書きします**（`<channel>_backup/` にバックアップを取ったうえで実行）。
ドライランの結果を確認してから実行すること。

In [ ]:
#@title ✍️ Phase 2: 整形 本実行（バックアップ付き上書き）
for name in config.CHANNEL_NAMES:
    maintain.align_schema(config.channel_dir(name), dry_run=False)

In [ ]:
#@title ✅ 整合性チェック（全フォルダ・全ファイル）
maintain.verify_schema()

In [ ]:
#@title ☁️ GCSアップロード（バケット: oshi-katsu / プレフィックス: youtube_stat）
from google.colab import auth as colab_auth
colab_auth.authenticate_user()

maintain.upload_to_gcs()